# Agent Communication and Coordination

## Northstar: investigate an EU checkout conversion drop

Three specialists contribute independent evidence about a 31% conversion drop after deployment `842`. A coordinator forms the smallest eligible team, a blackboard preserves attributable artifacts, a critic checks disagreement, and the system prepares—not executes—a mitigation proposal.

**Learning outcomes:** design typed messages and shared state; compare routing, delegation, handoffs, blackboards, debate, voting, and dynamic teams; contain conflict; and decide experimentally whether a team beats one bounded agent. **Safety boundary:** this is credential-free deterministic simulation; no model, network, or production action is invoked.

![Agent communication and coordination](../../../assets/agent-communication-coordination.svg)

Static SVG keeps the coordination diagram readable in GitHub and Jupyter. The shared board holds versioned evidence artifacts, not an unbounded chat transcript. Orange represents the convergence and human-control boundary.

## Step 1 — choose a team only when the task earns one

A single agent is the baseline. It is often cheaper, faster, and easier to evaluate. Form a team only if domains need different context/tools/permissions, independent work can run in parallel, or an independent critic catches material errors. The router is deterministic here: a cross-domain conversion incident forms three specialist roles; a simple request would remain a generalist workflow.

In [ ]:
"""Deterministic coordination controls for evidence-backed multi-agent work."""
from dataclasses import dataclass, field


@dataclass(frozen=True)
class Artifact:
    role: str
    claim: str
    source: str
    confidence: float


@dataclass
class TeamRun:
    required_roles: set[str] = field(default_factory=lambda: {"observability", "deployment", "impact"})
    artifacts: dict[str, Artifact] = field(default_factory=dict)
    trace: list[str] = field(default_factory=list)
    budget: int = 6
    status: str = "routed"


def assign(task: str) -> list[str]:
    """Deterministic router: create a team only for cross-domain incidents."""
    return ["observability", "deployment", "impact"] if "conversion" in task.lower() else ["generalist"]


def publish(run: TeamRun, artifact: Artifact) -> None:
    """Blackboard accepts one scoped, attributable artifact per authorized role."""
    if artifact.role not in run.required_roles:
        raise ValueError("unauthorized role")
    if not artifact.source or not 0 <= artifact.confidence <= 1:
        raise ValueError("artifact requires source and calibrated confidence")
    run.artifacts[artifact.role] = artifact
    run.trace.append(f"publish:{artifact.role}:{artifact.source}")


def converge(run: TeamRun) -> str:
    """A critic blocks disagreement; no majority vote can erase missing evidence."""
    if run.budget <= 0:
        run.status = "escalated"; run.trace.append("budget-exhausted"); return run.status
    if set(run.artifacts) != run.required_roles:
        run.status = "waiting"; run.trace.append("missing-artifact"); return run.status
    claims = [a.claim for a in run.artifacts.values()]
    if len(set(claims)) != 1 or min(a.confidence for a in run.artifacts.values()) < .7:
        run.status = "conflict"; run.trace.append("critic:request-evidence-or-human"); return run.status
    run.status = "proposal"; run.trace.append("consensus:proposal-only"); return run.status


def run_demo() -> TeamRun:
    run = TeamRun()
    for role, source in [("observability", "metrics-42"), ("deployment", "deploy-842"), ("impact", "sla-eu")]:
        publish(run, Artifact(role, "rollback deploy-842", source, .88))
    assert converge(run) == "proposal"
    return run


if __name__ == "__main__": print(run_demo())


In [1]:

task = 'Investigate a 31% EU checkout conversion decline after a deployment.'
roles = assign(task)
print(roles)
assert roles == ['observability', 'deployment', 'impact']

['observability', 'deployment', 'impact']


## Step 2 — communicate through typed, scoped artifacts

A role returns a claim, source, confidence, and—in production—tenant, correlation ID, evidence IDs, freshness, uncertainty, and an idempotency key. The system validates this before it reaches shared state. This avoids two frequent failures: agents treating each other's prose as authority, and an attractive conclusion losing its evidence trail.

This is a **blackboard** pattern: specialists publish artifacts; the coordinator and critic read a scoped view. It is not a voting machine and it does not permit arbitrary overwrite.

In [2]:
run = TeamRun()
for role, source in [('observability', 'metrics-42'), ('deployment', 'deploy-842'), ('impact', 'sla-eu')]:
    publish(run, Artifact(role, 'rollback deploy-842', source, .88))

print(run.artifacts)
print(*run.trace, sep='\n')
assert set(run.artifacts) == run.required_roles

{'observability': Artifact(role='observability', claim='rollback deploy-842', source='metrics-42', confidence=0.88), 'deployment': Artifact(role='deployment', claim='rollback deploy-842', source='deploy-842', confidence=0.88), 'impact': Artifact(role='impact', claim='rollback deploy-842', source='sla-eu', confidence=0.88)}
publish:observability:metrics-42
publish:deployment:deploy-842
publish:impact:sla-eu


## Step 3 — join, critique, and converge

A coordinator’s synthesis is only as good as the evidence it receives. A critic therefore checks required roles, source-backed claims, calibrated confidence, and conflict before a proposal exists. This is different from debate: debate generates counterarguments; the critic enforces a decision rule. Consensus needs a bounded rule and an escalation outcome. Majority agreement cannot compensate for missing evidence or correlated model error.

In [3]:
status = converge(run)
print(status, run.trace[-1])
assert status == 'proposal'

# Deliberate failure: conflicting evidence triggers a bounded conflict path, not more chatter.
conflict = TeamRun()
publish(conflict, Artifact('observability', 'rollback deploy-842', 'metrics-42', .9))
publish(conflict, Artifact('deployment', 'hold rollback; unrelated deploy', 'deploy-842', .9))
publish(conflict, Artifact('impact', 'rollback deploy-842', 'sla-eu', .9))
assert converge(conflict) == 'conflict'
print(conflict.trace[-1])

proposal consensus:proposal-only
critic:request-evidence-or-human


## Step 4 — delegation, handoffs, negotiation, and dynamic teams

**Delegation** creates a bounded subtask while the coordinator retains responsibility for integration. **Handoff** transfers conversational control and must carry a minimized, explicit context package plus a return/termination condition. **Negotiation** is appropriate for legitimate constrained trade-offs (for example, allocating a rate-limit budget), not for deciding facts by persuasion. **Dynamic team formation** is policy-constrained discovery: choose approved agents based on capability, tenant scope, tools, residency, availability, cost and conflicts of interest; cap team size and delegation depth.

Use an explicit task contract such as `role, objective, allowed_tools, expected_artifact, deadline, budget, tenant, correlation_id, stop_condition`. It prevents a vague instruction like *help investigate* from creating an unbounded delegation chain.

In [4]:
task_contract = {
    'role': 'deployment', 'objective': 'Assess deploy-842 contribution',
    'allowed_tools': ['read_deployment_history'], 'expected_artifact': 'source-backed claim',
    'tenant': 'northstar-eu', 'deadline_minutes': 5, 'budget_calls': 2,
    'stop_condition': 'publish artifact or escalate',
}
assert task_contract['allowed_tools'] == ['read_deployment_history']
assert task_contract['budget_calls'] <= 2
task_contract

{'role': 'deployment',
 'objective': 'Assess deploy-842 contribution',
 'allowed_tools': ['read_deployment_history'],
 'expected_artifact': 'source-backed claim',
 'tenant': 'northstar-eu',
 'deadline_minutes': 5,
 'budget_calls': 2,
 'stop_condition': 'publish artifact or escalate'}

## Step 5 — evaluate a team against the single-agent baseline

Do not compare different prompts, tools, budgets, or task sets. Hold those constant and record supported task success, evidence coverage, policy violations, cost per safe success, p95 latency, tool calls, coordination messages, conflict/escalation rate, and recovery behavior. Retain the team only if the gain is material for your task’s value and risk.

The synthetic result below shows a common outcome: teams can improve evidence coverage for cross-domain investigations while incurring communication cost. A simple lookup should not pay that price.

In [5]:
results = {
    'single_bounded_agent': {'supported_success': .76, 'cost': .018, 'p95_seconds': 5.9, 'messages': 1},
    'specialist_team': {'supported_success': .89, 'cost': .031, 'p95_seconds': 4.8, 'messages': 6},
}
for design, metric in results.items():
    metric['cost_per_safe_success'] = round(metric['cost'] / metric['supported_success'], 3)
    print(design, metric)

assert results['specialist_team']['supported_success'] > results['single_bounded_agent']['supported_success']
assert results['specialist_team']['cost'] > results['single_bounded_agent']['cost']

single_bounded_agent {'supported_success': 0.76, 'cost': 0.018, 'p95_seconds': 5.9, 'messages': 1, 'cost_per_safe_success': 0.024}
specialist_team {'supported_success': 0.89, 'cost': 0.031, 'p95_seconds': 4.8, 'messages': 6, 'cost_per_safe_success': 0.035}


## Production checklist, exercises, and references

- Bound team size, turns, messages, delegation depth, concurrency, token/tool budgets, and time; always define terminal escalation.
- Apply identity, tenant scope, least-privilege tools, provenance checks, schemas, idempotency, audit, and retention to every participant and message.
- Preserve minority reports and conflicts; do not hide them behind a synthesis or vote.
- Independently evaluate multi-agent paths against a strong single-agent and deterministic-workflow baseline.

**Exercises:** add stale-artifact rejection, a malicious discovered agent, an abstaining weighted vote, an approval packet, and a benchmark case where the single agent wins.

References: [LangGraph multi-agent patterns](https://docs.langchain.com/oss/python/langchain/multi-agent/index), [LangGraph subgraphs](https://docs.langchain.com/oss/python/langgraph/use-subgraphs), [AutoGen teams](https://microsoft.github.io/autogen/stable/user-guide/agentchat-user-guide/tutorial/teams.html), [A2A protocol](https://a2a-protocol.org/latest/), [Multi-Agent Collaboration Mechanisms survey](https://arxiv.org/abs/2501.06322), [AI Agent Systems survey](https://arxiv.org/abs/2601.01743).